In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
import os
import warnings

warnings.filterwarnings("ignore")

## <이항 로지스틱 회귀>

## 문제1

다음과 같은 GLM 이항분류모델을 만들고 결과를 확인합니다.

- penguins 데이터를 사용하여, 상수항을 포함한 GLM 모델을 생성합니다.
- train, test 데이터는 250개와 83개를 각각 사용합니다.
- 독립변수는 ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]를 사용합니다.
- 종속변수는 "sex"를 사용합니다.

In [83]:
url = "https://raw.githubusercontent.com/Soyoung-Yoon/bigdata/main/penguins.csv"

df = pd.read_csv(url)
df

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Chinstrap,Dream,49.7,18.6,195.0,3600.0,Male
1,Adelie,Dream,40.8,18.9,208.0,4300.0,Male
2,Gentoo,Biscoe,49.6,15.0,216.0,4750.0,Male
3,Gentoo,Biscoe,49.4,15.8,216.0,4925.0,Male
4,Chinstrap,Dream,50.5,19.6,201.0,4050.0,Male
...,...,...,...,...,...,...,...
328,Gentoo,Biscoe,43.5,14.2,220.0,4700.0,Female
329,Adelie,Dream,36.6,18.4,184.0,3475.0,Female
330,Adelie,Biscoe,40.5,17.9,187.0,3200.0,Female
331,Gentoo,Biscoe,43.8,13.9,208.0,4300.0,Female


In [84]:
def sep_sex(x):
    if x == "Male":
        return 1
    elif x == "Female":
        return 0

df["sex"] = df["sex"].apply(sep_sex)
df

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Chinstrap,Dream,49.7,18.6,195.0,3600.0,1
1,Adelie,Dream,40.8,18.9,208.0,4300.0,1
2,Gentoo,Biscoe,49.6,15.0,216.0,4750.0,1
3,Gentoo,Biscoe,49.4,15.8,216.0,4925.0,1
4,Chinstrap,Dream,50.5,19.6,201.0,4050.0,1
...,...,...,...,...,...,...,...
328,Gentoo,Biscoe,43.5,14.2,220.0,4700.0,0
329,Adelie,Dream,36.6,18.4,184.0,3475.0,0
330,Adelie,Biscoe,40.5,17.9,187.0,3200.0,0
331,Gentoo,Biscoe,43.8,13.9,208.0,4300.0,0


In [87]:
### 수동으로 로지스틱 회귀모델 적합
from statsmodels.api import GLM, families, add_constant

y = df["sex"]
X = df[["species", "island", "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]]

X = add_constant(X)
X = pd.get_dummies(X, columns=["species", "island"], drop_first=True, dtype="int")
X

model = GLM(y, X, family=families.Binomial()).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:                    sex   No. Observations:                  333
Model:                            GLM   Df Residuals:                      324
Model Family:                Binomial   Df Model:                            8
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -63.024
Date:                Mon, 25 Aug 2025   Deviance:                       126.05
Time:                        14:46:45   Pearson chi2:                     524.
No. Iterations:                     8   Pseudo R-squ. (CS):             0.6349
Covariance Type:            nonrobust                                         
=====================================================================================
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const               -80.3767     12.330     -6.519      0.000    -104.544     -56.210
bill_length_mm        0.6144      0.132      4.656      0.000       0.356       0.873
bill_depth_mm         1.6464      0.336      4.903      0.000       0.988       2.305
flipper_length_mm     0.0267      0.048      0.552      0.581      -0.068       0.121
body_mass_g           0.0058      0.001      5.351      0.000       0.004       0.008
species_Chinstrap    -7.4027      1.663     -4.452      0.000     -10.661      -4.144
species_Gentoo       -8.4276      2.597     -3.245      0.001     -13.518      -3.337
island_Dream          0.3242      0.809      0.401      0.689      -1.262       1.910
island_Torgersen     -0.5079      0.856     -0.593      0.553      -2.185       1.169
=====================================================================================
"""

In [89]:
model.llf

-63.023702642217586

In [ ]:
### 데이터 분할
train = df.iloc[:250,:]
test = df.iloc[250:,:]

In [65]:
### 로지스틱 회귀 모델 적합
from statsmodels.api import GLM, families

formula = "sex ~ bill_length_mm + bill_depth_mm + flipper_length_mm + body_mass_g"

model = GLM.from_formula(formula, data=train, family=families.Binomial()).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:                    sex   No. Observations:                  250
Model:                            GLM   Df Residuals:                      245
Model Family:                Binomial   Df Model:                            4
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -52.689
Date:                Mon, 25 Aug 2025   Deviance:                       105.38
Time:                        14:40:46   Pearson chi2:                     249.
No. Iterations:                     7   Pseudo R-squ. (CS):             0.6165
Covariance Type:            nonrobust                                         
=====================================================================================
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept           -69.6464     11.455     -6.080      0.000     -92.098     -47.195
bill_length_mm        0.0879      0.058      1.528      0.126      -0.025       0.201
bill_depth_mm         2.3989      0.344      6.972      0.000       1.725       3.073
flipper_length_mm     0.0038      0.042      0.091      0.928      -0.078       0.086
body_mass_g           0.0057      0.001      5.517      0.000       0.004       0.008
=====================================================================================
"""

In [66]:
### 모델 설명력 확인
model.llf

-52.68927559017419

In [67]:
### 설명력
model.pseudo_rsquared()

0.6164837599730447

In [8]:
print("로그-우도가 0에 가까울수록 모델 설명력이 좋다고 평가")

로그-우도가 0에 가까울수록 모델 설명력이 좋다고 평가


In [9]:
### 통계적으로 유의한 회귀계수 확인

model.pvalues[1:] > 0.05

bill_length_mm        True
bill_depth_mm        False
flipper_length_mm     True
body_mass_g          False
dtype: bool

In [10]:
### 모델이 통계적으로 유의한지 (우도비 검정)
from scipy.stats import chi2

# H0 : 모든 회귀계수가 0이다.(=모델이 통계적으로 유의하지 않다.)
# H1 : 적어도 하나의 회귀계수는 0이 아니다.(= 모델이 통계적으로 유의하다.)

# 우도비 검정통계량
s = model.null_deviance - model.deviance

# p-value
rv = chi2(df=model.df_model)

p = 1- rv.cdf(s)

print(f"우도비 검정통계량 : {s}")
print(f"우도비 검정통계량의 p-value : {p}")
print("귀무가설 채택" if p > 0.05 else "귀무가설 기각")

우도비 검정통계량 : 239.5933280488514
우도비 검정통계량의 p-value : 0.0
귀무가설 기각


In [11]:
print("""
모델이 통계적으로 유의하다.
""")


모델이 통계적으로 유의하다.



In [12]:
### 오즈비 해석
np.exp(model.params[1:])

bill_length_mm        1.091879
bill_depth_mm        11.011307
flipper_length_mm     1.003795
body_mass_g           1.005757
dtype: float64

In [13]:
### 예측 및 평가
from sklearn.metrics import accuracy_score

pred = model.predict(test).round().astype("int")

acc = accuracy_score(test["sex"], pred)
acc

0.8433734939759037

## 문제 2

다음과 같은 GLM 이항분류모델을 만들고 결과를 확인합니다.

- penguins 데이터를 사용하여, 상수항을 포함한 GLM 모델을 생성합니다.
- train, test 데이터는 250개와 83개를 각각 사용합니다.
- 독립변수는 ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "species", "island"]를 사용합니다.
- 종속변수는 "sex"를 사용합니다.

In [123]:
url = "https://raw.githubusercontent.com/Soyoung-Yoon/bigdata/main/penguins.csv"

df = pd.read_csv(url)
df

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Chinstrap,Dream,49.7,18.6,195.0,3600.0,Male
1,Adelie,Dream,40.8,18.9,208.0,4300.0,Male
2,Gentoo,Biscoe,49.6,15.0,216.0,4750.0,Male
3,Gentoo,Biscoe,49.4,15.8,216.0,4925.0,Male
4,Chinstrap,Dream,50.5,19.6,201.0,4050.0,Male
...,...,...,...,...,...,...,...
328,Gentoo,Biscoe,43.5,14.2,220.0,4700.0,Female
329,Adelie,Dream,36.6,18.4,184.0,3475.0,Female
330,Adelie,Biscoe,40.5,17.9,187.0,3200.0,Female
331,Gentoo,Biscoe,43.8,13.9,208.0,4300.0,Female


In [124]:
### 수동방식으로 로지스틱 회귀 모델 적합

def sep_sex(x):
    if x == "Male":
        return 1
    elif x == "Female":
        return 0

df["sex"] = df["sex"].apply(sep_sex)
df

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Chinstrap,Dream,49.7,18.6,195.0,3600.0,1
1,Adelie,Dream,40.8,18.9,208.0,4300.0,1
2,Gentoo,Biscoe,49.6,15.0,216.0,4750.0,1
3,Gentoo,Biscoe,49.4,15.8,216.0,4925.0,1
4,Chinstrap,Dream,50.5,19.6,201.0,4050.0,1
...,...,...,...,...,...,...,...
328,Gentoo,Biscoe,43.5,14.2,220.0,4700.0,0
329,Adelie,Dream,36.6,18.4,184.0,3475.0,0
330,Adelie,Biscoe,40.5,17.9,187.0,3200.0,0
331,Gentoo,Biscoe,43.8,13.9,208.0,4300.0,0


In [125]:
### 독립변수 종속변수 구분
from statsmodels.api import add_constant

y = df["sex"]
X = df[["species", "island", "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]]

# 더미 코딩
X = pd.get_dummies(X, columns=["species", "island"], drop_first=True, dtype="int")

# 상수항 추가
X = add_constant(X)
X

,const,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,species_Chinstrap,species_Gentoo,island_Dream,island_Torgersen
0,1.0,49.7,18.6,195.0,3600.0,1,0,1,0
1,1.0,40.8,18.9,208.0,4300.0,0,0,1,0
2,1.0,49.6,15.0,216.0,4750.0,0,1,0,0
3,1.0,49.4,15.8,216.0,4925.0,0,1,0,0
4,1.0,50.5,19.6,201.0,4050.0,1,0,1,0
...,...,...,...,...,...,...,...,...,...
328,1.0,43.5,14.2,220.0,4700.0,0,1,0,0
329,1.0,36.6,18.4,184.0,3475.0,0,0,1,0
330,1.0,40.5,17.9,187.0,3200.0,0,0,0,0
331,1.0,43.8,13.9,208.0,4300.0,0,1,0,0


In [126]:
# train, test 데이터는 250개와 83개를 각각 사용합
train_input = X.iloc[:250, :]
test_input = X.iloc[250:, :]

train_target = y.iloc[:250]
test_target = y.iloc[250:]

In [127]:
### 로지스틱 회귀모델 적합
from statsmodels.api import GLM, families

model = GLM(train_target, train_input, family=families.Binomial()).fit()

model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:                    sex   No. Observations:                  250
Model:                            GLM   Df Residuals:                      241
Model Family:                Binomial   Df Model:                            8
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -40.901
Date:                Mon, 25 Aug 2025   Deviance:                       81.803
Time:                        14:52:37   Pearson chi2:                     163.
No. Iterations:                     8   Pseudo R-squ. (CS):             0.6510
Covariance Type:            nonrobust                                         
=====================================================================================
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const               -97.4141     17.208     -5.661      0.000    -131.140     -63.688
bill_length_mm        0.5659      0.144      3.922      0.000       0.283       0.849
bill_depth_mm         2.1611      0.456      4.736      0.000       1.267       3.055
flipper_length_mm     0.0862      0.059      1.460      0.144      -0.030       0.202
body_mass_g           0.0054      0.001      3.950      0.000       0.003       0.008
species_Chinstrap    -7.4228      1.928     -3.850      0.000     -11.202      -3.644
species_Gentoo       -7.5983      3.056     -2.487      0.013     -13.587      -1.609
island_Dream          0.2288      1.008      0.227      0.820      -1.747       2.205
island_Torgersen     -0.6764      1.157     -0.584      0.559      -2.945       1.592
=====================================================================================
"""

In [128]:
### 모델의 설명력 (로그-우도)

model.llf

-40.90142551406656

In [131]:
### 모델의 설명력 (Pseudo r2)

model.pseudo_rsquared()

0.6509974718434134

In [20]:
## 모델의 통계적으로 유의한지 확인 (우도비 검정)
from scipy.stats import chi2

# 우도비 검정통계량
s = model.null_deviance - model.deviance

# 우도비 검정통계량의 p-value
rv = chi2(loc=0, scale=1, df=model.df_model)
p = 1- rv.cdf(s)

print(f"검정통계량 : {s}")
print(f"검정통계량의 p-value : {p}")
print("귀무가설 채택" if p > 0.05 else "귀무가설 기각")

검정통계량 : 263.1690282010667
검정통계량의 p-value : 0.0
귀무가설 기각


In [21]:
print("""
모델이 통계적으로 유의하다.
""")


모델이 통계적으로 유의하다.



In [22]:
## 예측 및 평가
from sklearn.metrics import f1_score

pred = np.where(model.predict(test_input) > 0.6, 1, 0)

f1 = f1_score(test_target, pred)
f1

0.8787878787878788

In [23]:
### 로그우도 구하기
model.llf

-40.90142551406656

In [24]:
### 잔차 이탈도 구하기
model.deviance

81.80285102813312

In [25]:
### 오즈비 해석
np.exp(model.params[1:])

bill_length_mm       1.761056
bill_depth_mm        8.681002
flipper_length_mm    1.090036
body_mass_g          1.005433
species_Chinstrap    0.000597
species_Gentoo       0.000501
island_Dream         1.257152
island_Torgersen     0.508452
dtype: float64

In [26]:
### 예측 및 평가
from sklearn.metrics import f1_score

pred = np.where(model.predict(test_input) > 0.6, 1, 0)

f1 = f1_score(test_target, pred)
f1

0.8787878787878788

In [27]:
### 예측확률의 신뢰구간 하한, 상한 구하기
result = model.get_prediction(test_input)

result.summary_frame(alpha=0.05)

,mean,mean_se,mean_ci_lower,mean_ci_upper
250,0.039746,0.030336,8.641260e-03,0.164266
251,0.006831,0.008069,6.678775e-04,0.066097
252,0.000010,0.000019,2.908854e-07,0.000368
253,0.995443,0.004550,9.683365e-01,0.999360
254,0.683671,0.191722,2.755102e-01,0.924717
...,...,...,...,...
328,0.005094,0.005063,7.221185e-04,0.035004
329,0.117481,0.077656,2.975561e-02,0.366218
330,0.087045,0.084651,1.168074e-02,0.434758
331,0.000129,0.000194,6.812202e-06,0.002440


## 다항 로지스틱 회귀

## 문제1 

다음 조건을 만족하는 모델을 생성하고, 로그-우도와 잔차이탈도를 출력하시오.

- 종속변수 : species
- 독립변수 : bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g
- tol 값을 10으로 사용

In [107]:
url = "https://raw.githubusercontent.com/Soyoung-Yoon/bigdata/main/penguins.csv"

df = pd.read_csv(url)
df

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Chinstrap,Dream,49.7,18.6,195.0,3600.0,Male
1,Adelie,Dream,40.8,18.9,208.0,4300.0,Male
2,Gentoo,Biscoe,49.6,15.0,216.0,4750.0,Male
3,Gentoo,Biscoe,49.4,15.8,216.0,4925.0,Male
4,Chinstrap,Dream,50.5,19.6,201.0,4050.0,Male
...,...,...,...,...,...,...,...
328,Gentoo,Biscoe,43.5,14.2,220.0,4700.0,Female
329,Adelie,Dream,36.6,18.4,184.0,3475.0,Female
330,Adelie,Biscoe,40.5,17.9,187.0,3200.0,Female
331,Gentoo,Biscoe,43.8,13.9,208.0,4300.0,Female


In [108]:
### 수동방식으로 다항 로지스틱 회귀 모델 적합
def sep_sepecies(x):
    if x == "Adelie":
        return 0
    elif x == "Chinstrap":
        return 1
    elif x == "Gentoo":
        return 2

df["species"] = df["species"].apply(sep_sepecies)
df

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,1,Dream,49.7,18.6,195.0,3600.0,Male
1,0,Dream,40.8,18.9,208.0,4300.0,Male
2,2,Biscoe,49.6,15.0,216.0,4750.0,Male
3,2,Biscoe,49.4,15.8,216.0,4925.0,Male
4,1,Dream,50.5,19.6,201.0,4050.0,Male
...,...,...,...,...,...,...,...
328,2,Biscoe,43.5,14.2,220.0,4700.0,Female
329,0,Dream,36.6,18.4,184.0,3475.0,Female
330,0,Biscoe,40.5,17.9,187.0,3200.0,Female
331,2,Biscoe,43.8,13.9,208.0,4300.0,Female


In [109]:
### 독립변수, 종속변수 나누기
y = df["species"]
X = df[["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]]

In [110]:
# 상수항 추가
from statsmodels.api import add_constant

X = add_constant(X)

In [111]:
### 데이터 분할 train : test == 250개 : 83개

train_input = X.iloc[:250, :]
test_input = X.iloc[250:, :]
train_target = y.iloc[:250]
test_target = y.iloc[250:]

In [112]:
### 다항 로지스틱 회귀 적합
from statsmodels.api import MNLogit

model = MNLogit(train_target, train_input).fit(tol=10)
model.summary()

Optimization terminated successfully.
         Current function value: 0.092967
         Iterations 2


<class 'statsmodels.iolib.summary.Summary'>
"""
                          MNLogit Regression Results                          
==============================================================================
Dep. Variable:                species   No. Observations:                  250
Model:                        MNLogit   Df Residuals:                      240
Method:                           MLE   Df Model:                            8
Date:                Mon, 25 Aug 2025   Pseudo R-squ.:                  0.9120
Time:                        14:50:14   Log-Likelihood:                -23.242
converged:                       True   LL-Null:                       -264.21
Covariance Type:            nonrobust   LLR p-value:                 5.286e-99
=====================================================================================
        species=1       coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const               -16.7712      9.368     -1.790      0.073     -35.133       1.590
bill_length_mm        0.6772      0.107      6.310      0.000       0.467       0.887
bill_depth_mm        -0.3015      0.230     -1.312      0.190      -0.752       0.149
flipper_length_mm     0.0026      0.054      0.048      0.961      -0.103       0.108
body_mass_g          -0.0022      0.001     -2.348      0.019      -0.004      -0.000
-------------------------------------------------------------------------------------
        species=2       coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const               -12.5480     11.622     -1.080      0.280     -35.326      10.230
bill_length_mm        0.3129      0.119      2.624      0.009       0.079       0.547
bill_depth_mm        -0.9426      0.258     -3.656      0.000      -1.448      -0.437
flipper_length_mm     0.0638      0.064      1.000      0.317      -0.061       0.189
body_mass_g           0.0004      0.001      0.389      0.697      -0.002       0.002
=====================================================================================
"""

In [113]:
### 모델의 설명력(로그-우도)

model.llf

-23.241681554856573

In [122]:
### McFadden’s Pseudo R²(모델 설명력)
model.prsquared

0.9120323567198771

In [116]:
### 잔차 이탈도

deviance = -2 * (model.llf - model.llnull)
deviance

-481.9309648914613

In [121]:
### 모델의 통계적 유의성 검정(우도비 검정)
from scipy.stats import chi2

# H0 : 모든 회귀계수가 0이다.(=모델이 통계적으로 유의하지 않다.)
# H1 : 적어도 하나의 회귀계수는 0이 아니다.(=모델이 통계적으로 유의하다.)

# 우도비 검정통계량
s = 2*(model.llf - model.llnull)

# 우도비 검정통계량의 p-value
rv = chi2(loc=0, scale=1, df=model.df_model)

p = 1-rv.cdf(s)

print(f"우도비 검정통계량 : {s}")
print(f"우도비 검정통계량의 p-value : {p}")
print("귀무가설 채택" if p > 0.05 else "귀무가설 기각")

우도비 검정통계량 : 481.9309648914613
우도비 검정통계량의 p-value : 0.0
귀무가설 기각


In [38]:
### 예측 및 평가
from sklearn.metrics import f1_score

pred = model.predict(test_input)
pred = np.argmax(pred.values, axis=1)

f1 = f1_score(test_target, pred, average="macro")
f1

0.9562476826103078

In [39]:
### formula 방식

In [40]:
url = "https://raw.githubusercontent.com/Soyoung-Yoon/bigdata/main/penguins.csv"

df = pd.read_csv(url)
df

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Chinstrap,Dream,49.7,18.6,195.0,3600.0,Male
1,Adelie,Dream,40.8,18.9,208.0,4300.0,Male
2,Gentoo,Biscoe,49.6,15.0,216.0,4750.0,Male
3,Gentoo,Biscoe,49.4,15.8,216.0,4925.0,Male
4,Chinstrap,Dream,50.5,19.6,201.0,4050.0,Male
...,...,...,...,...,...,...,...
328,Gentoo,Biscoe,43.5,14.2,220.0,4700.0,Female
329,Adelie,Dream,36.6,18.4,184.0,3475.0,Female
330,Adelie,Biscoe,40.5,17.9,187.0,3200.0,Female
331,Gentoo,Biscoe,43.8,13.9,208.0,4300.0,Female


In [41]:
### 원본 데이터
df

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Chinstrap,Dream,49.7,18.6,195.0,3600.0,Male
1,Adelie,Dream,40.8,18.9,208.0,4300.0,Male
2,Gentoo,Biscoe,49.6,15.0,216.0,4750.0,Male
3,Gentoo,Biscoe,49.4,15.8,216.0,4925.0,Male
4,Chinstrap,Dream,50.5,19.6,201.0,4050.0,Male
...,...,...,...,...,...,...,...
328,Gentoo,Biscoe,43.5,14.2,220.0,4700.0,Female
329,Adelie,Dream,36.6,18.4,184.0,3475.0,Female
330,Adelie,Biscoe,40.5,17.9,187.0,3200.0,Female
331,Gentoo,Biscoe,43.8,13.9,208.0,4300.0,Female


In [42]:
### 종속변수 숫자로 변환
def sep_species(x):
    if x == "Adelie":
        return 0
    elif x == "Chinstrap":
        return 1
    elif x == "Gentoo":
        return 2

df["species"] = df["species"].apply(sep_species)
df

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,1,Dream,49.7,18.6,195.0,3600.0,Male
1,0,Dream,40.8,18.9,208.0,4300.0,Male
2,2,Biscoe,49.6,15.0,216.0,4750.0,Male
3,2,Biscoe,49.4,15.8,216.0,4925.0,Male
4,1,Dream,50.5,19.6,201.0,4050.0,Male
...,...,...,...,...,...,...,...
328,2,Biscoe,43.5,14.2,220.0,4700.0,Female
329,0,Dream,36.6,18.4,184.0,3475.0,Female
330,0,Biscoe,40.5,17.9,187.0,3200.0,Female
331,2,Biscoe,43.8,13.9,208.0,4300.0,Female


In [43]:
### 데이터 분할 train : test == 250개 : 83개

train = df.iloc[:250, :]
test = df.iloc[250:, :]

In [44]:
### 다항 로지스틱회귀 모델 적합
from statsmodels.api import MNLogit

formula = "species ~ bill_length_mm + bill_depth_mm + flipper_length_mm + body_mass_g"

model = MNLogit.from_formula(formula, data=train).fit(tol=10)

model.summary()

Optimization terminated successfully.
         Current function value: 0.092967
         Iterations 2


<class 'statsmodels.iolib.summary.Summary'>
"""
                          MNLogit Regression Results                          
==============================================================================
Dep. Variable:                species   No. Observations:                  250
Model:                        MNLogit   Df Residuals:                      240
Method:                           MLE   Df Model:                            8
Date:                Mon, 25 Aug 2025   Pseudo R-squ.:                  0.9120
Time:                        13:52:57   Log-Likelihood:                -23.242
converged:                       True   LL-Null:                       -264.21
Covariance Type:            nonrobust   LLR p-value:                 5.286e-99
=====================================================================================
        species=1       coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept           -16.7712      9.368     -1.790      0.073     -35.133       1.590
bill_length_mm        0.6772      0.107      6.310      0.000       0.467       0.887
bill_depth_mm        -0.3015      0.230     -1.312      0.190      -0.752       0.149
flipper_length_mm     0.0026      0.054      0.048      0.961      -0.103       0.108
body_mass_g          -0.0022      0.001     -2.348      0.019      -0.004      -0.000
-------------------------------------------------------------------------------------
        species=2       coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept           -12.5480     11.622     -1.080      0.280     -35.326      10.230
bill_length_mm        0.3129      0.119      2.624      0.009       0.079       0.547
bill_depth_mm        -0.9426      0.258     -3.656      0.000      -1.448      -0.437
flipper_length_mm     0.0638      0.064      1.000      0.317      -0.061       0.189
body_mass_g           0.0004      0.001      0.389      0.697      -0.002       0.002
=====================================================================================
"""

In [45]:
### 우도비 검정
from scipy.stats import chi2

# H0 : 모든 회귀계수가 0이다.(=모델이 통계적으로 유의하지 않다.)
# H1 : 적어도 하나의 회귀계수는 0이 아니다.(=모델이 통계적으로 유의하다.)

# 우도비 검정통계량
s = 2*(model.llf - model.llnull)

# 우도비 검정통계량의 p-value
rv = chi2(loc=0, scale=1, df=model.df_model)

p = 1-rv.cdf(s)

print(f"우도비 검정통계량 : {s}")
print(f"우도비 검정통계량의 p-value : {p}")
print("귀무가설 채택" if p > 0.05 else "귀무가설 기각")

우도비 검정통계량 : 481.9309648914613
우도비 검정통계량의 p-value : 0.0
귀무가설 기각


In [46]:
### 회귀계수 통계적 유의성 검정

model.pvalues[1:].round(3)

,0,1
bill_length_mm,0.000,0.009
bill_depth_mm,0.190,0.000
flipper_length_mm,0.961,0.317
body_mass_g,0.019,0.697


In [47]:
### 예측하기
from sklearn.metrics import f1_score

# 예측확률 구하기
pred = model.predict(test)

# 확률이 가장 높은 종속변수값으로 변환
pred = np.argmax(pred.values, axis=1)

f1 = f1_score(test["species"], pred, average="macro")
f1

0.9562476826103078

In [48]:
### 성능평가
from sklearn.metrics import f1_score

f1 = f1_score(test["species"], pred, average="weighted")
f1

0.9627296728627525